### **GLOVAL-BibData:** A web-based bibliometric toolkit for systematic reviews, enabling data integration from Web of Science and Scopus, duplicate detection, bibliometric analysis, and PRISMA reporting.


*   **Created by:** Gloria Gómez and Valentina Carrillo

*   **Email:** gloriagomezosorio@gmail.com ; carrillo.vale@gmail.com

*   **Description:** A web-based bibliometric toolkit for systematic reviews, enabling data integration from Web of Science and Scopus, duplicate detection, bibliometric analysis, and PRISMA reporting.

🡆 **Install and import the libraries that will be used**

In [ ]:
!pip install networkx wordcloud openpyxl bibtexparser pycountry pandas matplotlib seaborn plotly pycountry rapidfuzz spacy
!python -m spacy download en_core_web_sm
!pip install plotly -q
import pandas as pd
import seaborn as sns
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import re
import unicodedata
import bibtexparser
import pycountry
import spacy
import math
import networkx as nx
import plotly.graph_objects as go
from rapidfuzz import fuzz
from matplotlib.patches import FancyBboxPatch, Rectangle, PathPatch
from matplotlib.lines import Line2D
from bibtexparser.model import DuplicateBlockKeyBlock
from scipy.interpolate import make_interp_spline
from collections import Counter
from scipy.interpolate import PchipInterpolator
from matplotlib.path import Path
from itertools import combinations, product
from wordcloud import WordCloud
import matplotlib.patheffects as pe
from matplotlib.lines import Line2D

import warnings
import logging

# Suppress pandas chained-assignment FutureWarnings
warnings.filterwarnings(
    "ignore",
    category=FutureWarning,
    message=".*ChainedAssignmentError.*"
)

# Suppress repeated BibTeX parser warnings
class BibtexWarningFilter(logging.Filter):

    def filter(self, record):
        return not (
            "Unknown block type" in record.getMessage()
        )

logging.getLogger().addFilter(BibtexWarningFilter())

# Install GLOVAL-BibData from GitHub Releases

%pip install https://github.com/carrillovale12/GLOVAL-BibData/releases/download/v0.1.0/gloval_bibdata-0.1.0-cp313-cp313-linux_x86_64.whl

# Import GLOVAL-BibData

from gloval_bibdata import (
    ReadDatabase,
    GlovalBibData,
    MergeDatabase,
    delete_duplicates,
    Prisma,
    ExploratoryAnalysis,
    donutplot,
    BarPlots,
    stacked_line_plot,
    AreaPlots,
    CumulativeKeywords,
    KeywordBursts,
    KeywordWordCloud,
    ProductionByYear,
    CooperationRelationships,
    SankeyFlows
)

print("GLOVAL-BibData imported successfully!")

🡆 **Available tools**

* **Database management:** reading, cleaning, standardization and integration of bibliographic records from Scopus and Web of Science.
* **PRISMA:** generation of PRISMA flow diagrams for systematic reviews.
* **Exploratory analysis:** exploration and descriptive analysis of bibliographic variables.
* **Visualization:** graphical representation of bibliometric indicators.
* **Keyword analysis:** analysis of keyword frequency, temporal evolution and keyword bursts.
* **Scientific production:** analysis of scientific production over time.
* **Network analysis:** visualization of cooperation relationships and relationships between bibliometric variables.

🡆 **Upload the .bib files from the Scopus or Web of Science databases.**


* The files should be named “Scopus” for Scopus and “Wos” for Web of Science.
* Also, upload the database: scimagojr_2025_area (Link: )

In [ ]:
from google.colab import files
uploaded = files.upload()

🡆 **Read the databases**
*   Description: read the database WOS first, then Scopus, and convert the entries into data frames. Return the total number of documents per database

In [ ]:
wos = ReadDatabase.read_bib("Wos.bib", col_origen="type")
scopus = ReadDatabase.read_bib("Scopus.bib", col_origen="type")
wos_num, scopus_num = ReadDatabase.count_docs(wos, scopus)

🡆 **Add variables to databases**

*   **Research area**: Supplement the files with SCIMAGOJR_2025 to determine the research area using the databases.
*   **Countries:** extraction of variables and function in each database extract the institution from the “affiliation” column and create a list of words associated with institutions, and then a data frame is created in each database.
*   **Institutions**: Extract the institution from the affiliation column and create a list with words associated with institutions

In [52]:
# extra_inst: an optional list of additional keywords for recognizing institutions;
# e.g.: 'observatorio', 'cirad'

# Load SCImago Journal Rank database

scimagojr_url = ("https://raw.githubusercontent.com/carrillovale12/GLOVAL-BibData/refs/heads/main/data/scimagojr_2025_area.csv")
scimagojr = pd.read_csv(scimagojr_url, sep=";",encoding="utf-8")

scopus, wos = GlovalBibData.process_data(scimagojr=scimagojr, scopus=scopus, wos=wos,
                                         extra_inst=['observatorio', 'cirad'])

🡆 **Merge databases**

* Description: process for merging the two databases; and counts the total number of records in the merged database

In [ ]:
scopus_std, wos_std = MergeDatabase.unify_databases(scopus, wos)
bases_with_dups, duplicates = MergeDatabase.merge_databases(None, None, wos_std, scopus_std)

🡆 **Remove duplicates by DOI and by title**

In [ ]:
database, duplicates, bases_without_dups, bases_with_dups = delete_duplicates(bases_with_dups)

🡆 **Export a consolidated database without duplicates in Excel format .xlsx**

In [26]:
# Export in Excel (.xlsx) format:
bases_without_dups.to_excel("merge_database.xlsx", index=False)

**PRISMA REPORTING**

* At this stage, you must decide whether a study meets the review’s inclusion criteria, which studies are eligible, and which are excluded based on the author’s criteria.
* Remove those that do not qualify from the database.
* Once finished, upload the database (merge_database.xlsx) again.

In [ ]:
from google.colab import files
uploaded = files.upload()

🡆 **Creation of the Prism Diagram**

*   Prism diagram (Identification, Screening, Eligibility, Included)
*   Download and View the diagram

In [ ]:
gloval = pd.read_excel("merge_database.xlsx")
prisma = Prisma.parameters(wos_num, scopus_num, bases_with_dups, database,
                           bases_without_dups, gloval,)
fig = prisma.prisma_diagram(save="prisma_diagram.png")

**BIBLIOMETRIC ANALYSIS**

🡆 **Exploratory data analysis**
*   Preparation of an exploratory table detailing the data for the variables under study

In [ ]:
gloval = pd.read_excel("merge_database.xlsx")
analysis = ExploratoryAnalysis(gloval)
df_variables = analysis.variable()
df_variables

🡆 **Graphs of bibliometric analysis**

* **Overview general**

**Types of Documents (Donut Chart)**

This donut chart shows the percentage share of each document type in your dataset, with the total count displayed in the center. What you can change:

* **width, height:** size of the figure.
* **min_gap:** minimum separation between neighboring percentages (increase if they overlap).
* **save:** 'None' to avoid saving.

In [ ]:
donutplot(data=gloval, min_gap=0.1,
          width=9, height=6, save='document_types.png')

**Documents by variable (Bar Chart, Top n)**

This bar chart shows the top (n) variable with the most publications, with the exact number of documents written inside each bar. What you can change:
* **var:** variable name ('year', 'countries', 'authors', 'journal', 'institutions',
                'areas').
* **top_n:** number of items to display (does not apply to 'year').
* **color:** bar color; **ticksbar:** y-axis interval size; **labelrotation:** rotation for the x-axis labels
* **width, height:** size of the figure; **save:** 'None' to avoid saving.

In [ ]:
# Documents by year
plots = BarPlots(analysis)
plots.barplot('year', color='#247d7f', labelrotation=0, va='bottom',
               width=10, height=4, ticksbar=20, save='docs_by_year.png')

# Documents by countries
plots = BarPlots(analysis, top_n=10)
plots.barplot('countries', color='#F54927', labelrotation=45,va='bottom',
               width=10, height=4, ticksbar=50, save='docs_by_countries.png')

# Documents by authors
plots = BarPlots(analysis, top_n=15)
plots.barplot('authors', color='#51613B', labelrotation=90,va='bottom'
              , width=10, height=4, ticksbar=5, save='docs_by_authors.png')

**Documents by variable (Horizontal Bar Chart, Top n)**

This horizontal bar chart shows the top (n) variables, with the exact number of documents shown at the end of each bar.
* **var:** variable name ('year', 'countries', 'authors', 'journal', 'institutions',
                'areas').
* **top_n:** number of items to display (does not apply to 'year').
* **color:** bar color; **ticksbar:** y-axis interval size; **labelrotation:** rotation for the x-axis labels
* **width, height:** size of the figure; **save:** 'None' to avoid saving.

In [ ]:
# Documents by research area
plots = BarPlots(analysis, top_n=20)
plots.horizontal_barplot(var='areas', color='#247d7f', ha='left',
               width=10, height=4, ticksbar = 25, save='docs_by_research_area.png')

# Documents by journal
plots = BarPlots(analysis, top_n=10)
plots.horizontal_barplot(var='journal', color='#5f4c8a', ha='left',
               width=10, height=4, ticksbar = 5, save='docs_by_journal.png')

**Documents per Year by Type + Average Citations (Stacked Bar + Line)**

This chart shows the number of documents published each year, split by document type (stacked bars), with a dashed line showing the average number of citations per year on a second axis. What you can change:
* **x_var** = x variable; **y_line** = y variable (line plot); **y_bar** = y variable (bar plot)
* **year_min:** first year to include; if None, uses the minimum value from the data.
* **year_max:** last year to include; if None, uses the maximum value from the data.
* **palette_color** = sns color palette
* **ticks_bar** = y-axis values ​​(bar plot), **ticks_line** = y-axis values ​​(line plot)
* **line_color** = line color (line plot)
* **width, height:** size of the figure; **save:** 'None' to avoid saving.

In [ ]:
stacked_line_plot(x_var = 'Year', y_line = 'Citation', y_bar = 'Document type', data = gloval,
                  palette_color = 'tab20', ticks_bar = 12, ticks_line = 20, line_color ='#3C4142',
                  width =12, height= 5, year_min=2020, year_max=None)

**Documents per Year; Contries (Area Line Chart)**

This chart shows the total number of documents published each year or contries as a line, with the area underneath shaded to highlight the growth trend over time.
* **var:** variable name ('year', 'countries', 'authors', 'journal', 'institutions',
                'areas').
* **top_n:** number of items to display (does not apply to 'year').
* **color:** area and line color; **ticksbar:** y-axis interval size; **labelrotation:** rotation for the x-axis labels
* **width, height:** size of the figure; **save:** 'None' to avoid saving.

In [ ]:
# Documents by year (area plot)
plots =AreaPlots(analysis)
plots.areaplot(var='year', data=gloval, color='#3dcf96', ticks_bar=10, labelrotation=0,
                 width=8, height=3, save='area_docs_by_year.png')

# Documents by countries (area plot)
plots = AreaPlots(analysis, top_n=10)
plots.areaplot(var='countries', data=gloval, color='#F54927', ticks_bar=50, labelrotation=90,
               width=8, height=4, save='area_docs_by_countries.png')

*   **Productivity**

**Cumulative Keywords per Year (Smoothed Line Chart)**

This chart shows how often the top (n) keywords have been used, accumulated year by year (You can select the years for the second graph), using smoothed curves to make the growth trend easier to see. What you can change:
* **top_n:** number of items to display.
* **year_min:** first year to include; if None, uses the minimum value from the data.
* **year_max:** last year to include; if None, uses the maximum value from the data.
* **ticks_bar** = y-axis values.
* **width, height:** size of the figure; **save:** 'None' to avoid saving.

In [ ]:
# Cumulative author keywords by year
kwa = CumulativeKeywords(data=gloval,ticksbar=25, top_n=10)
kwa.graph(save="kwa_cumulative.png")

In [ ]:
# Cumulative author keywords, defining the range of years
kwa = CumulativeKeywords(data=gloval, top_n=15, ticksbar=20, year_min = 2016, year_max=2026)
kwa.graph(save="kwa_2022_2026.png")

**Top Keywords with the Strongest Citation Bursts**

This chart detects keywords that had a sudden, sharp spike in usage during a specific period (using the Kleinberg burst detection algorithm), showing that period as an segment over the full study timeline. What you can change:
* **top_n:** number of items to display.
* **year_min:** first year to include; if None, uses the minimum value from the data.
* **year_max:** last year to include; if None, uses the maximum value from the data.
* **baseline_color:** full study range, in light blue.
* **burst_color:** color of the burst segment.
* **width, height:** size of the figure; **save:** 'None' to avoid saving.

In [ ]:
bursts = KeywordBursts(data=gloval, top_n=15)
bursts.graph(baseline_color='#c7c7c9', burst_color='#b23a3a',
             save="citation_bursts_15.png")

**Top (n) Production Over Time (Bubble Timeline)**

This chart shows each variable publication activity across years, with bubble size representing the number of documents that year, and the number displayed inside each bubble. What you can change:
* **variable:** variable name ('Countries', 'Authors', 'Journal', 'Institutions','Areas').
* **top_n:** number of items to display.
* **width, height:** size of the figure; **save:** 'None' to avoid saving.

In [ ]:
# Countries
countries = ProductionByYear(data=gloval, variable='Countries', top_n=10)
countries.graph(width=8,height=6, save='countries_production.png')

# Journal
journal = ProductionByYear(data=gloval, variable='Journal', top_n=15)
journal.graph(width=8,height=5, save='journal_production.png', year_min=2016, year_max=2026)

#Authors
authors= ProductionByYear(data=gloval, variable='Authors', top_n=5)
authors.graph(width=6,height=3, save='authors_production.png')

*  **Network diagrams**

**Keyword Co-occurrence Network**

This chart shows which keywords tend to appear together in the same documents: each circle is a keyword (sized by frequency), colored by topic cluster, with lines connecting keywords that co-occur (thicker lines = more co-occurrences). What you can change:
* **variable:** variable name ('Countries', 'Authors', 'Journal', 'Institutions','Areas').
* **centrality:**'degree', 'strength', 'betweenness', 'closeness', 'eigenvector', 'pagerank'.
* **min_doc:** the minimum frequency required to show a keyword (lower to show more keywords).
* **width, height:** size of the figure; **save:** 'None' to avoid saving.

In [ ]:
coop = CooperationRelationships(data=gloval, variable='Keywords')
coop.graph_network(min_docs=5, centrality='betweenness', save='network_coocurrence.png')

**Chord Diagram of Cooperation Relationships**

This chart shows collaboration between variable: each circle is an institution (sized by number of documents), colored by cluster group, with curved lines connecting institutions that co-published together. What you can change:
* **variable:** variable name ('Countries', 'Authors', 'Journal', 'Institutions','Areas').
* **centrality:**'degree', 'strength', 'betweenness', 'closeness', 'eigenvector', 'pagerank'.
* **min_doc:** the minimum frequency required to show a keyword (lower to show more keywords).
* **width, height:** size of the figure; **save:** 'None' to avoid saving.

In [ ]:
cooperation = CooperationRelationships(data=gloval, variable='Countries')
cooperation.graph_chord(min_docs=5, width= 6, height=6)

**Keyword Word Cloud**

This chart shows the most frequently used keywords in the dataset, with larger text meaning higher frequency, after merging different spelling/wording variants into a single normalized term. What you can change:
* **colormap** = matplotlib color palette (viridis, plasma, inferno, magma, cividis).
* **max_words:**  number of words shown inside WordCloud, lower this number to display only the
most frequent terms and reduce clutter.
* **width, height:** size of the figure; **save:** 'None' to avoid saving.

In [ ]:
wordcloud = KeywordWordCloud(gloval)
wordcloud.graph(max_words=100, colormap='plasma', save='keyword_wordcloud.png')

**Sankey Diagrams:** Three-Field Plot

This Sankey diagram shows the flow between three variables: which years are linked to which keywords, and which keywords are linked to which countries, with line thickness showing how many documents share that connection. What you can change:
* **variable:** variable name ('Countries', 'Authors', 'Journal', 'Institutions','Areas').
* **top_n:** number of items to display.
* **color_by:** 'origin' or 'destination'
* **palette:**  matplotlib color palette (ej: tab10, tab20, Set1, Set2, Set3, Dark2, Paired, Accent)
* **width, height:** size of the figure; **save:** 'None' to avoid saving.

In [ ]:
sankey= SankeyFlows(data=gloval, variables=('Year', 'Keywords', 'Countries'), top_n=5)
sankey.graph_sankey(color_by='origin', palette='Set3', show_values=True, width=12, height=6, save="sankey_diagram.png")

In [ ]:
sankey= SankeyFlows(data=gloval, variables=('Year', 'Keywords', 'Countries'), top_n= {'Year':5, 'Keywords':10, 'Countries':5})
sankey.graph_sankey(color_by='origin', palette='Set2', show_values=True, width=14, height=8, save="sankey_diagram_2.png")